<div style="width:100%; text-align:center; padding:10px 0;">
<img src="project_header.png" style="width:100%; max-width:100vw; height:auto; display:block; margin:0 auto;">
</div>

# EO Africa SWAM project 
## Create TSM data cube and timeseries

In this notebook we stack all the estimated TSM results and create a data cube containing the TSM timeseries 

#### This notebook does the following:
* creates a list of TSM nc output files to process
* Filter images by the number of valid pixels
* Do statistics to get the 10%, median, mean and 90% values for each image
* Plot the median TSM map
* Saves output to netcdf and csv

#### Requirements: 
* Make sure that you have already created some "*_TSM_*.nc" output files using the *TSM_Step3_calculate_TSM.ipynb* notebook.

#### Settings to adjust manually:
* Manually define the lake/dam name in code cell 4 (options: ZV, RV, TW, MV, VV, CW), default is TW. 

### Version history:
* Version 1.0, 23 Feb 2026

##### Authors:
**Dalin Jiang**, University of Stirling, UK

In [1]:
import glob
import os
from datetime import datetime
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib.colors as colors

## Step 1: Search for the TSM files for one lake
set up the following:
* input path: the TSM results of each date separately
* output path: where the results of data cube and timeseries will be saved

In [2]:
# set up the input and output path
#
in_path = "/media/DATA/TSM/BV/"
out_path = "/media/DATA/TSM_cube/"

all_img = sorted(glob.glob(in_path+'*.nc'))
lake_name = os.path.basename(all_img[0]).split("_")[4]
print(f'found {len(all_img)} images, with the first 5 showing below:')
all_img[0:5]

found 665 images, with the first 5 showing below:


['/home/eoafrica/shared/SWAM/final_processing/TSM/TW_new/S2A_20150810T084916_T34HCH_C2XC_TW_TSM.nc',
 '/home/eoafrica/shared/SWAM/final_processing/TSM/TW_new/S2A_20150929T081736_T34HCH_C2XC_TW_TSM.nc',
 '/home/eoafrica/shared/SWAM/final_processing/TSM/TW_new/S2A_20151019T082012_T34HCH_C2XC_TW_TSM.nc',
 '/home/eoafrica/shared/SWAM/final_processing/TSM/TW_new/S2A_20151208T082332_T34HCH_C2XC_TW_TSM.nc',
 '/home/eoafrica/shared/SWAM/final_processing/TSM/TW_new/S2A_20151218T082342_T34HCH_C2XC_TW_TSM.nc']

In [ ]:
# open all data, and concat them together
#
ds = xr.open_mfdataset(all_img,
                       combine='by_coords',
                       parallel=True,
                       data_vars=['TSM']
                      )

sdate = ds.time.dt.strftime('%Y-%m-%d').isel(time=0).item()
edate = ds.time.dt.strftime('%Y-%m-%d').isel(time=-1).item()
#print(ds.shape)

## Step 2: Filter images by the number of valid pixels
This processing will filter the data using the number of valid pixels of each image, i.e., these images with less than a certain number of valid pixels (here 5%) will be excluded to avoid outliers.

In [ ]:
# Step 1: Calculate valid water pixels count per day
valid_pixels_per_day = ds['TSM'].notnull().sum(dim=['lat', 'lon'])
max_water = valid_pixels_per_day.max().values

# Step 2: Total pixels per image
total_pixels = ds['TSM'].sizes['lat'] * ds.sizes['lon']

# Step 3: Define threshold for minimum 5% valid water pixels
threshold = 0.05 * max_water

# Step 4: Create mask to select only days where valid pixels >= threshold
valid_days_mask = valid_pixels_per_day >= threshold

# Step 5: Filter data to those valid days
ds_valid = ds.sel(time=valid_days_mask)
num_ts = ds_valid['TSM'].notnull().sum(dim=['lat','lon'])

print(f' number of valid images after filtering: {num_ts.shape}')

## Step 3: Calculate the 10%, mean, median and 90% values of each image
This processing will calcualte the 10%, mean, median and 90% values of TSM for each image, this results in a TSM timeseries, results will be exported to a csv file

In [ ]:
# calculate the median, 10% and 90% data, as well as monthly median data
#
p10_ts = ds_valid['TSM'].reduce(np.nanpercentile, q=10, dim=['lat', 'lon'])               # 10% of all valid images, timeseries
p90_ts = ds_valid['TSM'].reduce(np.nanpercentile, q=90, dim=['lat', 'lon'])               # 90% of all valid images, timeseries
median_ts = ds_valid['TSM'].median(dim=['lat', 'lon'], skipna=True)                       # median of all valid images, timeseries
mean_ts = ds_valid['TSM'].mean(dim=['lat', 'lon'], skipna=True)                           # mean of all valid images, timeseries

median_map = ds_valid.median(dim=['time'], skipna=True)                                   # median of all valid images, map
monthly_median = ds_valid['TSM'].groupby('time.month').median(dim=['time'], skipna=True)  # monthly median of all valid images

## Step 4: Plot the median TSM map

In [ ]:
# plot the median TSM map
#
fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={'projection': ccrs.PlateCarree()})

# Plot median TSM as colored raster
img = ax.pcolormesh(median_map['lon'], median_map['lat'], median_map['TSM'], 
                    shading='auto', cmap='viridis', vmin=0, vmax=20, transform=ccrs.PlateCarree())

ax.coastlines()
ax.gridlines(draw_labels=True)

cbar = fig.colorbar(img, ax=ax, orientation='vertical', label='Median TSM (g/m3)')

plt.title('Median TSM over Time')
plt.tight_layout()
# plt.savefig(output+lake_name+'_median_TSM_alltime_over_validpix05.png',dpi=300)  # uncomment this line to export the figure

plt.show()

## Step 5: Plot the TSM time series

In [ ]:
# Plot timeseries with shaded percentile range
plt.figure(figsize=(12, 6))
plt.plot(median_ts['time'], median_ts, label='Median Daily TSM')
plt.fill_between(median_ts['time'], p10_ts, p90_ts, color='gray', alpha=0.3,
                 label='10th–90th Percentile Range')
plt.xlabel('Date')
plt.ylabel('TSM (g/m3)')
plt.title(f'Daily Median TSM for dates with > 5% valid water pixels')
plt.legend()
plt.grid(True)
plt.tight_layout()
# plt.savefig(output+lake_name+'_daily_median_percentiles_TSM_over_time_validpix05.png',dpi=300)  # uncomment this line to export the figure

plt.show()

## Step 6: Export results
The following reults will be exported:
* the statistics of TSM in Step 3 to a csv file
* the data cube, which is a 3D data (lat x lon x time) including all TSM data

In [ ]:
# export results to disk: 
# 1. TSM data cube
ds = ds.chunk({"time":10,"lat":500, "lon":500})
ds.attrs = {'Product': 'Total Suspended Matter',
            'Units': 'g m^-3',
            'Sensor': 'Sentinel2-MSI',
            'AC Algorithm': 'C2XComplex',
            'AC Algorithm DOI': 'Brockmann et al. (2016)',
            'TSM algorithm': 'Jiang et al. (2023)',
            'TSM algorithm DOI': 'https://doi.org/10.1016/j.isprsjprs.2023.09.020',
            'Project': 'ESA EO Africa R&D SWAM',
            'Project PI':'Marie Smith, CSIR; Dalin Jiang, University of Stirling',
            'PI contact':'msmith2@csir.co.za; dalin.jiang@stir.ac.uk',
            'History': str(datetime.utcnow()) + ' Python',
            'Startdate':sdate,
            'Enddate':edate,
            'Lake_name':lake_name,
            'Lake_location':'Western Cape, South Africa',
            'Min_Latitude':ds.lat.min().item(),
            'Max_Latitude':ds.lat.max().item(),
            'Min_Longitude':ds.lon.min().item(),
            'Max_Longitude':ds.lon.max().item(),
            }

comp = dict(zlib=True, complevel=6, dtype='float32')  # 1–9 (higher = more compression, slower)
encoding = {var: comp for var in ds.data_vars}

out_cube_file = out_path+"TSM_cube_2015_2025_"+lake_name+".nc"
ds.to_netcdf(out_cube_file,format='NETCDF4', encoding=encoding) # engine="h5netcdf" for faster writing

In [ ]:
# merge all statistics, assume all data have the same order by time
mg_dt = xr.merge([p10_ts.rename("TSM_10P"), 
                  mean_ts.rename("TSM_mean"),
                  median_ts.rename("TSM_median"),
                  p90_ts.rename("TSM_90P"),
                  num_ts.rename("N_valid")])

# convert to dataframe
tmp_df = mg_dt.to_dataframe()

# add time and lake information
tmp_df["Date"] = tmp_df.index.map(lambda x: datetime.date(x))
tmp_df["DateTime_UTC"] = tmp_df.index
tmp_df["Lake_name"] = lake_name

# reorder the data
ts_df = tmp_df[["Lake_name","DateTime_UTC","Date", "TSM_10P","TSM_mean","TSM_median","TSM_90P","N_valid"]]

# export the data to a csv
out_ts_file = out_path+"TSM_timeseries_2015_2025_"+lake_name+".csv"
ts_df.to_csv(out_ts_file, index=False)